# Spotify Music Recommendation System
## Notebook 03 — Data Cleaning & Preprocessing

**Purpose:** Fix all known data quality issues found in notebook 01, and save clean datasets to `data/processed/`.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Raw Data](#2-load-raw-data)
3. [Fix 1 — Clean song_and_artists](#3-fix-1--clean-song_and_artists)
4. [Fix 2 — Parse artist list strings](#4-fix-2--parse-artist-list-strings)
5. [Fix 3 — Handle hidden-empty genres](#5-fix-3--handle-hidden-empty-genres)
6. [Fix 4 — release_date and type fixes](#6-fix-4--release_date-and-type-fixes)
7. [Fix 5 — Duration and outlier flags](#7-fix-5--duration-and-outlier-flags)
8. [Final Snapshot](#8-final-snapshot)
9. [Save Cleaned Datasets](#9-save-cleaned-datasets)
10. [Summary](#10-summary)

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import ast
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

DATA_DIR   = Path('../data/raw')
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory ready:', OUTPUT_DIR.resolve())

## 2. Load Raw Data

In [ ]:
data       = pd.read_csv(DATA_DIR / 'data.csv')
data_genre = pd.read_csv(DATA_DIR / 'data_w_genres.csv')
genre_df   = pd.read_csv(DATA_DIR / 'data_by_genres.csv')
year_df    = pd.read_csv(DATA_DIR / 'data_by_year.csv')
song_df    = pd.read_csv(DATA_DIR / 'song_and_artists.csv')

print('Loaded raw datasets:')
for name, df in [('data',data),('data_genre',data_genre),('genre_df',genre_df),
                  ('year_df',year_df),('song_df',song_df)]:
    print(f'  {name:<15} {df.shape}')

## 3. Fix 1 — Clean song_and_artists

**Problems:** 16 duplicate rows, 10 null `Singer/Artists`, 3 null `Album/Movie`, and `User-Rating` stored as a string format `'9.4/10'`.

In [ ]:
print('Before cleaning:')
print('  Rows      :', len(song_df))
print('  Duplicates:', song_df.duplicated().sum())
print('  Nulls     :', song_df.isnull().sum().to_dict())

In [ ]:
# Step 1a: Remove duplicates
song_clean = song_df.drop_duplicates().reset_index(drop=True)

# Step 1b: Fill minor nulls
song_clean['Singer/Artists'] = song_clean['Singer/Artists'].fillna('Unknown')
song_clean['Album/Movie']    = song_clean['Album/Movie'].fillna('Unknown')

# Step 1c: Parse User-Rating '9.4/10' -> 9.4 (float)
song_clean['rating'] = pd.to_numeric(
    song_clean['User-Rating'].str.replace('/10', '', regex=False),
    errors='coerce'
)

# Step 1d: Rename columns to snake_case
song_clean = song_clean.rename(columns={
    'Song-Name'     : 'song_name',
    'Singer/Artists': 'artist',
    'Genre'         : 'genre',
    'Album/Movie'   : 'album',
    'User-Rating'   : 'user_rating_raw',
})

print('After cleaning:')
print('  Rows      :', len(song_clean))
print('  Duplicates:', song_clean.duplicated().sum())
print('  Nulls     :', song_clean.isnull().sum().to_dict())
print('  Rating range:', song_clean['rating'].min(), '–', song_clean['rating'].max())
song_clean.head(3)

## 4. Fix 2 — Parse artist list strings

In `data.csv`, the `artists` column is stored as a Python list string like `"['Artist A', 'Artist B']"`. We parse it to get a real list, then create a clean comma-separated display string.

> **Note:** `artists` in `data_w_genres.csv` is already a plain string — no parsing needed there.

In [ ]:
def parse_artists(s):
    """Parse "['A', 'B']" -> ['A', 'B']. Returns [s] on failure."""
    try:
        result = ast.literal_eval(str(s))
        return result if isinstance(result, list) else [str(result)]
    except Exception:
        return [str(s)]

data['artists_list']  = data['artists'].apply(parse_artists)
data['artists_clean'] = data['artists_list'].apply(lambda x: ', '.join(x))
data['n_artists']     = data['artists_list'].apply(len)

print('Parsing done.')
print(f'Single-artist tracks : {(data["n_artists"] == 1).sum():,}')
print(f'Multi-artist tracks  : {(data["n_artists"] > 1).sum():,}')
print()
# Show before / after
data[['artists', 'artists_clean', 'n_artists']].head(5)

## 5. Fix 3 — Handle hidden-empty genres

In `data_w_genres.csv`, the `genres` column is a Python list string. Rows with `'[]'` are **not null** but have no genre. We must parse and filter them.

In [ ]:
def parse_genres(s):
    """Parse '["hip hop", "pop"]' -> ['hip hop', 'pop']. Returns [] on failure or []."""
    try:
        result = ast.literal_eval(str(s))
        return result if isinstance(result, list) else []
    except Exception:
        return []

data_genre['genres_list']  = data_genre['genres'].apply(parse_genres)
data_genre['genres_clean'] = data_genre['genres_list'].apply(lambda x: ', '.join(x))
data_genre['n_genres']     = data_genre['genres_list'].apply(len)
data_genre['has_genre']    = data_genre['n_genres'] > 0

# Show real counts
total = len(data_genre)
has   = data_genre['has_genre'].sum()
empty = total - has
print(f'Total artists          : {total:,}')
print(f'Artists WITH genres    : {has:,}  ({has/total*100:.1f}%)')
print(f'Artists WITHOUT genres : {empty:,}  ({empty/total*100:.1f}%)  <- these had [] string')
print()
data_genre[['artists', 'genres', 'genres_clean', 'n_genres', 'has_genre']].head(6)

In [ ]:
# Also fix the 1 empty-genre row in data_by_genres.csv
problem = (genre_df['genres'] == '[]').sum()
print(f'data_by_genres: {problem} row has genres=[] — dropping it')
genre_df_clean = genre_df[genre_df['genres'] != '[]'].reset_index(drop=True)
print(f'Rows after drop: {len(genre_df_clean)}')

## 6. Fix 4 — release_date and type fixes

- `release_date` has 3 formats — extract year cleanly and use `year` column instead
- Cast `explicit` → bool, `key` and `mode` → int
- Drop the constant `mode` column from `data_by_year` (it is always 1)

In [ ]:
# Parse release_date -> release_year (handles all 3 formats)
data['release_year'] = (
    pd.to_datetime(data['release_date'], errors='coerce')
      .dt.year
      .fillna(data['year'])      # fall back to year column if parse fails
      .astype(int)
)

# Type fixes
data['explicit'] = data['explicit'].astype(bool)
data['key']      = data['key'].astype(int)
data['mode']     = data['mode'].astype(int)

# Drop constant mode from year_df
year_df_clean = year_df.drop(columns=['mode'])

print('Type fixes applied.')
print('year_df mode column dropped. Remaining columns:', year_df_clean.columns.tolist())
print()
# Check year vs release_year agreement
match = (data['year'] == data['release_year']).mean()
print(f'year == release_year agreement: {match*100:.1f}%')

## 7. Fix 5 — Duration and outlier flags

We add `duration_min` and flag extreme tracks without deleting them.
Only very short tracks (< 1 min) are removed from the modelling subset — they are interludes or sound effects, not real songs.

In [ ]:
data['duration_min']    = (data['duration_ms'] / 60_000).round(3)
data['flag_short']      = data['duration_min'] < 1.0
data['flag_long']       = data['duration_min'] > 30.0
data['flag_zero_pop']   = data['popularity'] == 0

print('Outlier flags added:')
print(f'  Tracks < 1 min    : {data["flag_short"].sum():,}')
print(f'  Tracks > 30 min   : {data["flag_long"].sum():,}')
print(f'  Tracks pop = 0    : {data["flag_zero_pop"].sum():,}')

# Clean modelling set: remove interludes
data_clean = data[~data['flag_short']].reset_index(drop=True)
print(f'\nModelling dataset after removing < 1 min tracks: {len(data_clean):,}')

## 8. Final Snapshot

In [ ]:
print('=== data_clean ===')
print(f'Shape: {data_clean.shape}')
print('Columns:', data_clean.columns.tolist())
print()
print('=== song_clean ===')
print(f'Shape: {song_clean.shape}')
print('Columns:', song_clean.columns.tolist())
print()
print('=== data_genre (genres parsed) ===')
print(f'Shape: {data_genre.shape}')
print(f'Artists with real genres: {data_genre["has_genre"].sum():,}')

## 9. Save Cleaned Datasets

In [ ]:
data_clean.to_csv(OUTPUT_DIR / 'data_clean.csv', index=False)
song_clean.to_csv(OUTPUT_DIR / 'song_and_artists_clean.csv', index=False)
year_df_clean.to_csv(OUTPUT_DIR / 'data_by_year_clean.csv', index=False)
genre_df_clean.to_csv(OUTPUT_DIR / 'data_by_genres_clean.csv', index=False)

# Save artist-genre mapping (only artists that have genres)
artist_genre_map = (
    data_genre[data_genre['has_genre']][['artists', 'genres_clean', 'n_genres']]
    .reset_index(drop=True)
)
artist_genre_map.to_csv(OUTPUT_DIR / 'artist_genres.csv', index=False)

print('Saved to data/processed/:')
for f in OUTPUT_DIR.iterdir():
    print(' ', f.name)

## 10. Summary

| Fix # | Problem | Solution | Rows Affected |
|---|---|---|---|
| 1 | `song_and_artists` — 16 duplicates | `drop_duplicates()` | −16 |
| 1 | `song_and_artists` — 13 null values | Fill with 'Unknown' | 13 |
| 1 | `User-Rating` string format | Parse to float | 2,420 |
| 2 | `artists` in `data.csv` — list strings | `ast.literal_eval()` + join | 170,653 |
| 3 | `genres` in `data_w_genres` — hidden `[]` | Parse + `has_genre` flag | 9,857 |
| 3 | `genres` in `data_by_genres` — 1 empty row | Drop row | −1 |
| 4 | `release_date` mixed formats | Parse → `release_year` column | 1,610 |
| 4 | `mode` in `data_by_year` — constant 1 | Drop column | — |
| 4 | Type mismatches | Cast `explicit`→bool, `key`/`mode`→int | — |
| 5 | Tracks < 1 minute | Remove from modelling set | −~150 |

**Next:** `04_feature_engineering.ipynb` — build the 14-dimensional audio feature vector.